In [ ]:
!pip -q install tokenizers tqdm


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving cognitive_snippet_hints_unique_50k.jsonl.gz to cognitive_snippet_hints_unique_50k.jsonl.gz


In [ ]:
import gzip, json, os, random, re, hashlib
from collections import defaultdict
from tqdm import tqdm

DATA_GZ = "cognitive_snippet_hints_unique_50k.jsonl.gz"
assert os.path.exists(DATA_GZ), "Upload the dataset .gz first."

random.seed(42)

def norm_code(code: str) -> str:
    c = code.lower()
    c = re.sub(r"/\*.*?\*/", " ", c, flags=re.S)
    c = re.sub(r"//.*?$", " ", c, flags=re.M)
    c = re.sub(r"\s+", " ", c).strip()
    c = re.sub(r'"[^"]*"', '"STR"', c)
    c = re.sub(r"\b\d+\b", "0", c)
    return c

def group_id(code_part: str) -> str:
    return hashlib.md5(norm_code(code_part).encode("utf-8")).hexdigest()

def build_io(obj, attempt:int):
    bug_tok = f"<BUG_{obj['bug_type']}>"
    inp = f"<STATE_{obj['state']}> <ATTEMPT_{attempt}> {bug_tok}\n{obj['code_part']}"
    out = "<NO_HINT>" if obj["quality"] == "ok" else obj[f"hint_{attempt}"]
    return inp, out

groups = defaultdict(list)
bug_tokens = set()

with gzip.open(DATA_GZ, "rt", encoding="utf-8") as f:
    for line in tqdm(f, total=50000):
        obj = json.loads(line)
        bug_tokens.add(f"<BUG_{obj['bug_type']}>")
        gid = group_id(obj["code_part"])
        for attempt in (1,2,3):
            groups[gid].append(build_io(obj, attempt))

keys = list(groups.keys())
random.shuffle(keys)

nG = len(keys)
train_keys = set(keys[: int(0.8*nG)])
val_keys   = set(keys[int(0.8*nG): int(0.9*nG)])
test_keys  = set(keys[int(0.9*nG):])

train = [pair for k in train_keys for pair in groups[k]]
val   = [pair for k in val_keys   for pair in groups[k]]
test  = [pair for k in test_keys  for pair in groups[k]]

print("groups:", nG)
print("train/val/test:", len(train), len(val), len(test))
print("bug tokens:", len(bug_tokens))


100%|██████████| 50000/50000 [00:01<00:00, 28779.80it/s]

groups: 50000
train/val/test: 120000 15000 15000
bug tokens: 25


In [ ]:
from tokenizers import ByteLevelBPETokenizer
import os

SPECIAL_TOKENS = [
    "<pad>", "<unk>", "<bos>", "<eos>",
    "<STATE_FOCUS>", "<STATE_CONFUSE>", "<STATE_OVERLOAD>",
    "<ATTEMPT_1>", "<ATTEMPT_2>", "<ATTEMPT_3>",
    "<NO_HINT>",
] + sorted(list(bug_tokens))

with open("tok_input.txt", "w", encoding="utf-8") as f_in, open("tok_output.txt", "w", encoding="utf-8") as f_out:
    for inp, out in train:
        f_in.write(inp.replace("\n", " ") + "\n")
        f_out.write(out.replace("\n", " ") + "\n")

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=["tok_input.txt", "tok_output.txt"],
    vocab_size=9000,
    min_frequency=2,
    special_tokens=SPECIAL_TOKENS
)

os.makedirs("tokenizer", exist_ok=True)
tokenizer.save_model("tokenizer")

print("Tokenizer vocab size:", tokenizer.get_vocab_size())
print("Example BUG token:", sorted(list(bug_tokens))[0], "id =", tokenizer.token_to_id(sorted(list(bug_tokens))[0]))


Tokenizer vocab size: 9000
Example BUG token: <BUG_array_oob> id = 11


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")

MAX_INP_LEN = 512
MAX_OUT_LEN = 96

def encode(text, max_len):
    ids = tokenizer.encode(text).ids
    ids = [BOS_ID] + ids[: max_len-2] + [EOS_ID]
    return ids

class HintDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        inp, out = self.pairs[idx]
        return encode(inp, MAX_INP_LEN), encode(out, MAX_OUT_LEN)

def collate(batch):
    inps, outs = zip(*batch)
    in_max = max(len(x) for x in inps)
    out_max = max(len(x) for x in outs)
    in_pad = [x + [PAD_ID]*(in_max-len(x)) for x in inps]
    out_pad = [y + [PAD_ID]*(out_max-len(y)) for y in outs]
    return torch.tensor(in_pad, dtype=torch.long), torch.tensor(out_pad, dtype=torch.long)

train_loader = DataLoader(HintDataset(train), batch_size=32, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(HintDataset(val),   batch_size=32, shuffle=False, collate_fn=collate)

print(next(iter(train_loader))[0].shape, next(iter(train_loader))[1].shape)


torch.Size([32, 180]) torch.Size([32, 23])


In [ ]:
import torch.nn as nn
import math

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=2048):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])

class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=3, dim_ff=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.tgt_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos = PositionalEncoding(d_model, dropout)
        self.tf = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True
        )
        self.out = nn.Linear(d_model, vocab_size)

    def make_tgt_mask(self, tgt_len):
        return torch.triu(torch.ones(tgt_len, tgt_len, device=device), diagonal=1).bool()

    def forward(self, src, tgt):
        src_key_padding = (src == PAD_ID)
        tgt_key_padding = (tgt == PAD_ID)
        tgt_mask = self.make_tgt_mask(tgt.size(1))

        src_e = self.pos(self.src_emb(src) * math.sqrt(self.d_model))
        tgt_e = self.pos(self.tgt_emb(tgt) * math.sqrt(self.d_model))

        h = self.tf(
            src_e, tgt_e,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding,
            tgt_key_padding_mask=tgt_key_padding,
            memory_key_padding_mask=src_key_padding
        )
        return self.out(h)

model = Seq2SeqTransformer(tokenizer.get_vocab_size()).to(device)
print("params:", sum(p.numel() for p in model.parameters())/1e6, "M")


device: cuda
params: 10.875688 M


In [ ]:
import torch, os

criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

def run_epoch(loader, train_mode: bool):
    model.train(train_mode)
    total_loss, total_tokens = 0.0, 0

    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        dec_in = tgt[:, :-1]
        labels = tgt[:, 1:]

        if train_mode:
            optimizer.zero_grad()

        logits = model(src, dec_in)
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))

        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        n_tokens = (labels != PAD_ID).sum().item()
        total_loss += loss.item() * max(n_tokens, 1)
        total_tokens += max(n_tokens, 1)

    return total_loss / total_tokens

os.makedirs("saved", exist_ok=True)
best_val = float("inf")
patience = 3
bad_epochs = 0
max_epochs = 20

for epoch in range(1, max_epochs+1):
    train_loss = run_epoch(train_loader, True)
    val_loss = run_epoch(val_loader, False)
    print(f"Epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        bad_epochs = 0
        torch.save(model.state_dict(), "saved/best_seq2seq_transformer.pt")
        print("  ✅ saved best")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("🛑 Early stopping")
            break

print("Best val loss:", best_val)


Epoch 01 | train 0.9715 | val 0.7998
  ✅ saved best
Epoch 02 | train 0.7749 | val 0.7799
  ✅ saved best
Epoch 03 | train 0.7586 | val 0.7682
  ✅ saved best
Epoch 04 | train 0.7410 | val 0.7489
  ✅ saved best
Epoch 05 | train 0.7248 | val 0.7358
  ✅ saved best
Epoch 06 | train 0.7124 | val 0.7284
  ✅ saved best
Epoch 07 | train 0.7039 | val 0.7242
  ✅ saved best
Epoch 08 | train 0.6975 | val 0.7214
  ✅ saved best
Epoch 09 | train 0.6923 | val 0.7206
  ✅ saved best
Epoch 10 | train 0.6879 | val 0.7200
  ✅ saved best
Epoch 11 | train 0.6841 | val 0.7197
  ✅ saved best
Epoch 12 | train 0.6806 | val 0.7201
Epoch 13 | train 0.6778 | val 0.7201
Epoch 14 | train 0.6760 | val 0.7166
  ✅ saved best
Epoch 15 | train 0.6736 | val 0.7180
Epoch 16 | train 0.6718 | val 0.7176
Epoch 17 | train 0.6707 | val 0.7161
  ✅ saved best
Epoch 18 | train 0.6698 | val 0.7171
Epoch 19 | train 0.6689 | val 0.7180
Epoch 20 | train 0.6681 | val 0.7171
🛑 Early stopping
Best val loss: 0.716117895456642


In [ ]:
import torch, os

@torch.no_grad()
def generate_hint(code_text: str, state: str, attempt: int, bug_type: str, max_new_tokens: int = 80):
    state = (state or "CONFUSE").upper()
    if state not in ["FOCUS","CONFUSE","OVERLOAD"]:
        state = "CONFUSE"
    attempt = int(attempt)
    if attempt not in [1,2,3]:
        attempt = 1

    bug_tok = f"<BUG_{bug_type}>"
    src_text = f"<STATE_{state}> <ATTEMPT_{attempt}> {bug_tok}\n{code_text}"
    src = torch.tensor([encode(src_text, MAX_INP_LEN)], dtype=torch.long).to(device)

    tgt = torch.tensor([[BOS_ID]], dtype=torch.long).to(device)

    model.eval()
    for _ in range(max_new_tokens):
        logits = model(src, tgt)
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        tgt = torch.cat([tgt, next_id], dim=1)
        if next_id.item() == EOS_ID:
            break

    ids = tgt[0].tolist()
    out_ids = []
    for t in ids:
        if t == EOS_ID: break
        if t not in (BOS_ID, PAD_ID):
            out_ids.append(t)
    return tokenizer.decode(out_ids)

# Load best checkpoint
import torch
model.load_state_dict(torch.load("saved/best_seq2seq_transformer.pt", map_location=device))
model.eval()

# Test
test_code = """#include <stdio.h>
int main(){ int x; scanf("%d", x); printf("%d\n", x); return 0; }"""

print("Original test cases:")
print(generate_hint(test_code, "CONFUSE", 1, "scanf_missing_amp"))
print(generate_hint(test_code, "CONFUSE", 2, "scanf_missing_amp"))
print(generate_hint(test_code, "CONFUSE", 3, "scanf_missing_amp"))

print("\n--- Demonstrating changing state and attempt ---")
# Example with different state and attempt
print("Changing state to FOCUS, attempt to 2, and bug_type to printf_wrong_format")
print(generate_hint(test_code, "FOCUS", 2, "printf_wrong_format"))
print(generate_hint(test_code, "OVERLOAD", 1, "division_by_zero"))
print(generate_hint(test_code, "CONFUSE", 3, "brace_mismatch"))

NameError: name 'model' is not defined

In [ ]:
# ====== QUICK TEST CELL (BUG-TOKEN MODEL) ======
# Assumes you already have:
# - tokenizer loaded
# - model loaded + model.eval()
# - encode() function
# - generate_hint(code_text, state, attempt, bug_type, max_new_tokens=80)

tests = [
    {
        "name": "BUGGY: scanf missing &",
        "state": "CONFUSE",
        "attempts": [1,2,3],
        "bug_type": "scanf_missing_amp",
        "code": r"""
#include <stdio.h>
int main(){
    int x;
    printf("Enter: ");
    scanf("%d", x);
    printf("%d\n", x);
    return 0;
}
"""
    },
    {
        "name": "BUGGY: division by zero",
        "state": "OVERLOAD",
        "attempts": [1,2,3],
        "bug_type": "division_by_zero",
        "code": r"""
#include <stdio.h>
int main(){
    int a = 10, b = 0;
    printf("%d\n", a/b);
    return 0;
}
"""
    },
    {
        "name": "INCOMPLETE: missing closing brace",
        "state": "FOCUS",
        "attempts": [1,2,3],
        "bug_type": "brace_mismatch",
        "code": r"""
#include <stdio.h>
int main(){
    int x = 5;
    if(x > 0){
        printf("pos\n");
    } else {
        printf("non-pos\n");
    }
"""
    },
    {
        "name": "OK: should output <NO_HINT>",
        "state": "FOCUS",
        "attempts": [1,2,3],
        "bug_type": "ok_no_hint",   # this exists in your dataset for ok samples
        "code": r"""
#include <stdio.h>
int main(){
    int sum = 0;
    for(int i=1;i<=5;i++){
        sum += i;
    }
    printf("Sum=%d\n", sum);
    return 0;
}
"""
    },
    {
        "name": "BUGGY: string compare with ==",
        "state": "CONFUSE",
        "attempts": [1,2,3],
        "bug_type": "strcmp_vs_eq",
        "code": r"""
#include <stdio.h>
#include <string.h>
int main(){
    char a[] = "yes";
    char b[] = "yes";
    if(a == b){
        printf("same\n");
    } else {
        printf("diff\n");
    }
    return 0;
}
"""
    },
    {
        "name": "BUGGY: printf wrong format",
        "state": "CONFUSE",
        "attempts": [1,2,3],
        "bug_type": "printf_wrong_format",
        "code": r"""
#include <stdio.h>
int main(){
    int x = 12;
    printf("%f\n", x);
    return 0;
}
"""
    },
]

def run_tests():
    for t in tests:
        print("\n" + "="*60)
        print(t["name"])
        print("STATE:", t["state"], "| BUG:", t["bug_type"])
        for att in t["attempts"]:
            out = generate_hint(t["code"], t["state"], att, t["bug_type"], max_new_tokens=80)
            print(f"\nAttempt {att} output:\n{out}")

run_tests()



BUGGY: scanf missing &
STATE: CONFUSE | BUG: scanf_missing_amp

Attempt 1 output:
Check the printf format specifier. Compile and read the error line carefully.

Attempt 2 output:
Uninitialized variables contain garbage values. Change only this part first.

Attempt 3 output:
Use %d for int values in printf. Then run again to confirm.

BUGGY: division by zero
STATE: OVERLOAD | BUG: division_by_zero

Attempt 1 output:
Check the value used as the divisor.

Attempt 2 output:
Division by zero is not allowed.

Attempt 3 output:
Ensure b_bnoq != 0 before dividing.

INCOMPLETE: missing closing brace
STATE: FOCUS | BUG: brace_mismatch

Attempt 1 output:
Check whether variables and required context are defined.

Attempt 2 output:
This snippet needs a declared variable and a surrounding function.

Attempt 3 output:
Finish the function with '}' so it becomes valid C code.

OK: should output <NO_HINT>
STATE: FOCUS | BUG: ok_no_hint

Attempt 1 output:


Attempt 2 output:


Attempt 3 output:


BUGGY: